# Brute-Force IND (Inclusion Dependency) Discovery Baseline - DVDRental

Reimplements the core logic of Metanome's BINDER algorithm in pure Python.
For each pair of columns across different tables, checks if `col_A ⊆ col_B`.

**Pipeline:**
1. Load all columns from all CSVs in `./data/`
2. For each cross-table column pair, check unary inclusion dependency
3. Apply cardinality filter (FK side ≤ PK side)
4. Filter to reference only ground-truth PK columns (from `./sql_ddl.sql`)
5. Compare against DDL ground truth

**Expected results:** TP=9, FP=26, FN=9, P=25.7%, R=50.0%, F1=0.340

In [1]:
import os
import re
import glob
import time
import pandas as pd
import numpy as np
from collections import defaultdict


def clean_table_name(name):
    name = re.sub(r'^asm_', '', name)
    name = re.sub(r'_\d{12,}$', '', name)
    return name.strip('_')


def parse_ddl_ground_truth(ddl_path):
    with open(ddl_path) as f:
        ddl = f.read()

    pk_gt = {}
    current_table = None
    for line in ddl.split('\n'):
        m = re.match(r'CREATE TABLE\s+"?(\w+)"?', line, re.I)
        if m:
            current_table = m.group(1).lower()
        pk_match = re.search(r'PRIMARY KEY\s*\(([^)]+)\)', line, re.I)
        if pk_match and current_table:
            cols = tuple(c.strip().strip('"').lower() for c in pk_match.group(1).split(','))
            pk_gt[current_table] = cols

    fk_gt = set()
    current_table = None
    for line in ddl.split('\n'):
        m = re.match(r'CREATE TABLE\s+"?(\w+)"?', line, re.I)
        if m:
            current_table = m.group(1).lower()
        fk_match = re.search(
            r'FOREIGN KEY\s*\((\w+)\)\s*REFERENCES\s+"?(\w+)"?\s*\((\w+)\)', line, re.I
        )
        if fk_match and current_table:
            fk_gt.add((
                current_table,
                fk_match.group(1).lower(),
                fk_match.group(2).lower(),
                fk_match.group(3).lower(),
            ))
    return pk_gt, fk_gt


def discover_inds_bruteforce(data_folder, inclusion_threshold=0.95):
    print("Loading all columns...")
    files = sorted(glob.glob(os.path.join(data_folder, "*.csv")))
    table_columns = {}

    for fpath in files:
        table = os.path.basename(fpath).replace('.csv', '')
        try:
            try:
                df = pd.read_csv(fpath, on_bad_lines='skip', engine='pyarrow')
            except:
                df = pd.read_csv(fpath, on_bad_lines='skip', low_memory=False)
        except Exception as e:
            print(f"  WARN: Could not read {fpath}: {e}")
            continue

        col_vals = {}
        for col in df.columns:
            vals = df[col].dropna().astype(str).str.strip().str.lower()
            col_vals[col] = set(vals.unique())
        table_columns[table] = col_vals
        print(f"  {clean_table_name(table)}: {len(df)} rows, {len(col_vals)} columns")

    tables = list(table_columns.keys())
    n_pairs = 0
    inds = []

    print(f"\nChecking inclusion dependencies across {len(tables)} tables...")
    t0 = time.time()

    for i, tbl_a in enumerate(tables):
        for col_a, vals_a in table_columns[tbl_a].items():
            if not vals_a:
                continue
            for tbl_b in tables:
                if tbl_a == tbl_b:
                    continue
                for col_b, vals_b in table_columns[tbl_b].items():
                    if not vals_b:
                        continue
                    n_pairs += 1

                    overlap = len(vals_a & vals_b)
                    inclusion = overlap / len(vals_a)
                    card_ok = len(vals_a) <= len(vals_b) * 1.05

                    if inclusion >= inclusion_threshold and card_ok:
                        inds.append({
                            'dep_table': tbl_a,
                            'dep_col': col_a,
                            'ref_table': tbl_b,
                            'ref_col': col_b,
                            'inclusion': inclusion,
                            'dep_distinct': len(vals_a),
                            'ref_distinct': len(vals_b),
                        })

    elapsed = time.time() - t0
    print(f"  Checked {n_pairs:,} column pairs in {elapsed:.1f}s")
    print(f"  Found {len(inds)} raw INDs (inclusion >= {inclusion_threshold})")

    return inds


def filter_inds_to_fks(inds, pk_gt=None):
    if pk_gt:
        pk_cols = set()
        for table, cols in pk_gt.items():
            for col in cols:
                pk_cols.add((table, col))

        filtered = []
        for ind in inds:
            ref_clean = clean_table_name(ind['ref_table']).lower()
            if (ref_clean, ind['ref_col'].lower()) in pk_cols:
                filtered.append(ind)
        inds = filtered
        print(f"  After PK filter: {len(inds)} INDs reference PK columns")

    best = {}
    for ind in inds:
        key = (ind['dep_table'], ind['dep_col'])
        if key not in best or ind['inclusion'] > best[key]['inclusion']:
            best[key] = ind

    results = sorted(best.values(), key=lambda x: -x['inclusion'])
    print(f"  After dedup: {len(results)} FK candidates")
    return results


def evaluate_inds(ind_fks, fk_gt):
    discovered = set()
    for ind in ind_fks:
        src_table = clean_table_name(ind['dep_table']).lower()
        src_col = ind['dep_col'].lower()
        tgt_table = clean_table_name(ind['ref_table']).lower()
        tgt_col = ind['ref_col'].lower()
        discovered.add((src_table, src_col, tgt_table, tgt_col))

    tp = len(discovered & fk_gt)
    fp = len(discovered - fk_gt)
    fn = len(fk_gt - discovered)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'tp': tp, 'fp': fp, 'fn': fn,
        'precision': precision, 'recall': recall, 'f1': f1,
        'discovered': discovered,
    }


##### =========== RUN =========== #####
data_folder = "./data"
ddl_path = "./sql_ddl.sql"
name = "DVDRental"

print(f"\n{'='*65}")
print(f"  IND BASELINE (BINDER-equivalent) - {name}")
print(f"{'='*65}\n")

pk_gt, fk_gt = parse_ddl_ground_truth(ddl_path)
print(f"Ground truth: {len(pk_gt)} PKs, {len(fk_gt)} FKs\n")

raw_inds = discover_inds_bruteforce(data_folder, inclusion_threshold=0.95)
fk_candidates = filter_inds_to_fks(raw_inds, pk_gt)
metrics = evaluate_inds(fk_candidates, fk_gt)

print(f"\n{'='*65}")
print(f"  IND BASELINE RESULTS - {name}")
print(f"{'='*65}")
print(f"  TP={metrics['tp']}  FP={metrics['fp']}  FN={metrics['fn']}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1:        {metrics['f1']:.4f}")

if metrics['fn'] > 0:
    print(f"\n  Missed FKs (false negatives):")
    for fk in sorted(fk_gt - metrics['discovered']):
        print(f"    {fk[0]}.{fk[1]} -> {fk[2]}.{fk[3]}")

if metrics['fp'] > 0:
    print(f"\n  False positives:")
    for fk in sorted(metrics['discovered'] - fk_gt):
        print(f"    {fk[0]}.{fk[1]} -> {fk[2]}.{fk[3]}")


  IND BASELINE (BINDER-equivalent) - DVDRental

Ground truth: 15 PKs, 18 FKs

Loading all columns...
  language: 6 rows, 3 columns
  actor: 200 rows, 4 columns
  address: 603 rows, 8 columns
  category: 16 rows, 3 columns
  city: 600 rows, 4 columns
  country: 109 rows, 3 columns
  customer: 599 rows, 10 columns
  film: 1000 rows, 13 columns
  film_actor: 5462 rows, 3 columns
  film_category: 1000 rows, 3 columns
  inventory: 4581 rows, 4 columns
  payment: 14596 rows, 6 columns
  rental: 16044 rows, 7 columns
  staff: 2 rows, 11 columns
  store: 2 rows, 4 columns

Checking inclusion dependencies across 15 tables...
  Checked 6,592 column pairs in 0.0s
  Found 547 raw INDs (inclusion >= 0.95)
  After PK filter: 330 INDs reference PK columns
  After dedup: 35 FK candidates

  IND BASELINE RESULTS - DVDRental
  TP=9  FP=26  FN=9
  Precision: 0.2571
  Recall:    0.5000
  F1:        0.3396

  Missed FKs (false negatives):
    city.country_id -> country.country_id
    film_category.categor